In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math
import random as rn
import numpy as np

In [3]:
# Use a quantum circuit to generate a random bit.
def get_random_basis(possible_basis):
    q1 = QuantumCircuit(1, 1)
    theta = 2*np.arccos(1/np.sqrt(3))
    
    #put into 1/sqrt(3)|0> + sqrt(2/3)|1>
    q1.ry(theta, 0)
    #measure first qubit
    q1.measure(0,0)

    result1 = run_circuit(q1)
    if result1 == '0':
        return possible_basis[0]

    #apply hadamard to the second qubit to split it 
    q2 = QuantumCircuit(1, 1)
    q2.h(0)
    q2.measure(0, 0)

    result2 = run_circuit(q2)
    if result2 == '0':
        return possible_basis[1]  
    else:
        return possible_basis[2]
    

# Construct a circuit that prepares the state  1/sqrt(2)( |01> - |10> )
def entangledPair():
    q = QuantumCircuit(2,2)
    q.h(0)
    q.cx(0,1)
    q.x(1)
    q.z(0)
    return q


def measure_basis(qc, operator, qubit):
  if operator == "X":
    qc.h(qubit)
  elif operator == "W":
    qc.ry(-np.pi/4, qubit)
  elif operator == "V":
    qc.ry(np.pi/4, qubit)
  #no changes for Z
  return qc


def run_circuit(qc):
  backend = BasicSimulator()
  compiled = transpile(qc, backend)
  job = backend.run(compiled, shots=1)
  result = job.result()
  counts = result.get_counts(compiled)
  bits = list(counts.keys())[0]
  return bits


def conv_bit(bit):
  #convert 0 to1 and 1 to -1 for S calculation
  if bit == "0":
    return 1
  elif bit == "1":
    return -1


In [8]:
N=100 #key length

#define the basises that allice and bob can measure in
alice_basis = ["X","W","Z"]
bob_basis = ["W","Z","V"]

#lists to store measurements, basis choices and results
alice_results = []
bob_results = []
alice_key = []
bob_key = []
othersA = []
othersB = []

for i in range((9*N)//2):
  #Alice and bob each receive one qubit of an entablged pair in state 1/sqrt(2)(|01⟩ − |10⟩)
  qc = entangledPair()

  #Alice randomly chooses an operator from her list
  alice_choice = get_random_basis(alice_basis)

  #Bob randomly chooses an operator from his list
  bob_choice = get_random_basis(bob_basis)

  #Alice measures operator Ai on her qubit.
  measure_basis(qc, alice_choice, 1)

  #Bob measures operator Bj on his qubit
  measure_basis(qc, bob_choice, 0)

  qc.measure([0,1],[0,1])
  result = run_circuit(qc)

  alice_bit = int(result[1])
  bob_bit = int(result[0])
    
  # we need to flip bob's bit 
  bob_bit = 1 - bob_bit

  #store alice and bobs results and basis choices
  alice_results.append((alice_choice, alice_bit))
  bob_results.append((bob_choice, bob_bit))

#Alice and Bob now compare their basis choices
for i in range(len(alice_results)):
  if (alice_results[i][0] == "W" and bob_results[i][0] == "W") or (alice_results[i][0] == "Z" and bob_results[i][0] == "Z"):
    alice_key.append(alice_results[i][1])
    bob_key.append(bob_results[i][1])

  else:
    #store results for non mathcing basis to calculate S
    othersA.append((alice_results[i]))
    othersB.append((bob_results[i]))

#key should be roughly length n as they should match 2/9 times.
print(f"N = {N}")
print(f"key length = {len(alice_key)}")

#we can see that alice and bobs secret keys match
print("alice key = ")
for i in range(len(alice_key)):
  print(alice_key[i], end="")

print("\nbob key = ")
for i in range(len(alice_key)):
  print(bob_key[i], end="")



N = 100
key length = 105
alice key = 
011111111100110001011011100100000100001100001001010100010110001111110010000000101010001010011001000001000
bob key = 
011111111100110001011011100100000100001100001001010100010110001111110010000000101010001010011001000001000

In [9]:
#entanglement test
XorW = []
XorV = []
ZorW = []
ZorV = []

for i in range(len(othersA)):
  #convert 0 to +1 and 1 to -1
  result = conv_bit(str(othersA[i][1])) * conv_bit(str(othersB[i][1]))

  if str(othersA[i][0]) == "X":
    if othersB[i][0] == "W":
      XorW.append(result)
    elif othersB[i][0] == "V":
      XorV.append(result)

  elif othersA[i][0] == "Z":
    if othersB[i][0] == "W":
      ZorW.append(result)
    elif othersB[i][0] == "V":
      ZorV.append(result)
  #other combinations are discarded

#calculate averages
xw = sum(XorW) / len(XorW)
xv = sum(XorV) / len(XorV)
zw = sum(ZorW) / len(ZorW)
zv = sum(ZorV) / len(ZorV)

#S =|⟨X⊗W⟩−⟨X⊗V⟩+⟨Z⊗W⟩+⟨Z⊗V⟩|
S = abs(xw - xv + zw + zv)
print(f"S = {S}")

if S <= 2:
  print("complete loss of quantum correlation: attacker detected")
else:
  print("no attacker detected")




S = 2.8558912359376314
no attacker detected
